In [1]:
import numpy as np
from scipy.stats import ttest_1samp, shapiro, wilcoxon, t, ttest_ind, ttest_rel, wilcoxon, binomtest, nct, norm
from statsmodels.stats.power import TTestPower

## 📌 Question 1 : Paired t-Test (Blood Pressure)

Blood pressure (mmHg) measured before and after medication for 8 patients:  
Before: `[140, 138, 150, 145, 142, 148, 135, 152]`  
After: `[132, 130, 142, 138, 135, 140, 128, 144]`  
At α = 0.05, test whether the medication significantly reduces blood pressure. Also compute a 95% CI for the mean reduction.

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean of (before - after) differences.

$$H_0: \mu_d = 0 \quad \text{(no change)}$$
$$H_1: \mu_d > 0 \quad \text{(blood pressure decreased, one-tailed)}$$

#### Step 2 : Why this test

The same 8 patients are measured twice, so the observations are paired. A paired t-test reduces between-subject noise by working directly on the within-subject differences. With the differences treated as a single sample, this collapses to a one-sample t-test on $d_i = \text{before}_i - \text{after}_i$. Population variance is unknown, so t (not Z) is correct.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d}$ is the mean difference, $s_d$ is the sample standard deviation of the differences.

In [2]:
before = np.array([140, 138, 150, 145, 142, 148, 135, 152])  # before readings
after  = np.array([132, 130, 142, 138, 135, 140, 128, 144])  # after readings
alpha  = 0.05                                                  # significance level
n      = len(before)                                           # sample size

In [3]:
d     = before - after          # paired differences
d_bar = d.mean()                # mean difference
sd    = np.std(d, ddof=1)       # sample std of differences
se    = sd / np.sqrt(n)         # standard error
df    = n - 1                   # degrees of freedom
t_stat = d_bar / se             # test statistic

#### Step 4 : Critical Value Method

In [4]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 41.671
t_crit : 1.895


**t_stat (41.67) > t_crit (1.895) → Reject H₀**

#### Step 5 : p-value Method

In [5]:
pval = t.sf(t_stat, df)                # one-tailed p-value (upper tail)
print(f'p-val : {pval:.12f}')

p-val : 0.000000000598


**p-val ≈ 0.0000000006 < 0.05 → Reject H₀**

#### Step 6 : Confidence Interval

A 95% CI for $\mu_d$ uses the two-tailed critical value:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [6]:
t_ci  = t.ppf(1 - alpha/2, df)         # two-tailed critical value for CI
moe   = t_ci * se                       # margin of error
lo    = d_bar - moe                     # lower bound
hi    = d_bar + moe                     # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (7.192, 8.058)


#### ✅ Final Conclusion

| Metric | Value |
|--|--|
| Mean difference $\bar{d}$ | 7.625 mmHg |
| Std deviation $s_d$ | 0.518 |
| Standard error | 0.183 |
| t statistic | 41.671 |
| t critical (one-tailed) | 1.895 |
| p-value | ≈ 0.0000000006 |
| 95% CI for $\mu_d$ | (7.192, 8.058) |
| Decision | Reject H₀ |

Strong evidence that the medication reduces blood pressure. The mean reduction is about 7.6 mmHg, and the 95% CI (7.19, 8.06) sits entirely above zero, confirming the effect is real and practically meaningful.

---

## 📌 Question 2 : Paired t-Test + Effect Size (VO₂ Max)

VO₂ max differences after a 12-week fitness program for 10 athletes:  
`[+4.2, +3.8, +5.1, +2.9, +6.0, +3.3, +4.7, +5.5, +2.1, +4.4]`  
Test at α = 0.05 (one-tailed) whether the program improved VO₂ max.  
Also compute Cohen's $d_z = \bar{d} / s_d$ and interpret the effect size.

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean improvement in VO₂ max.

$$H_0: \mu_d = 0 \quad \text{(no improvement)}$$
$$H_1: \mu_d > 0 \quad \text{(program improved VO₂ max, one-tailed)}$$

#### Step 2 : Why this test

The differences are already provided directly (before minus after is implicit). Each value is the within-subject change for one athlete, so this is still a paired design. With $n = 10$ and unknown population variance, a one-sample t-test on the differences is appropriate. One-tailed because the question only asks about improvement.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

In [7]:
d      = np.array([4.2, 3.8, 5.1, 2.9, 6.0, 3.3, 4.7, 5.5, 2.1, 4.4])  # VO2 max improvements
n      = len(d)                                                             # sample size
d_bar  = d.mean()                                                           # mean improvement
sd     = np.std(d, ddof=1)                                                  # sample std
se     = sd / np.sqrt(n)                                                    # standard error
df     = n - 1                                                              # degrees of freedom
t_stat = d_bar / se                                                         # test statistic

#### Step 4 : Critical Value Method

In [8]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 11.009
t_crit : 1.833


**t_stat (11.009) > t_crit (1.833) → Reject H₀**

#### Step 5 : p-value Method

In [9]:
pval = t.sf(t_stat, df)                # one-tailed p-value (upper tail)
print(f'p-val : {pval:.12f}')

p-val : 0.000000799635


**p-val ≈ 0.0000008 < 0.05 → Reject H₀**

#### Step 6 : Confidence Interval

95% CI for $\mu_d$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [10]:
t_ci  = t.ppf(1 - alpha/2, df)         # two-tailed critical value for CI
moe   = t_ci * se                       # margin of error
lo    = d_bar - moe                     # lower bound
hi    = d_bar + moe                     # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (3.337, 5.063)


#### Part (b) : Cohen's $d_z$ (Effect Size)

$$d_z = \frac{\bar{d}}{s_d}$$

Benchmarks: 0.2 = small, 0.5 = medium, 0.8 = large.

In [11]:
d_z = d_bar / sd                        # Cohen's d_z for paired design
print(f"Cohen's d_z : {d_z:.3f}")

Cohen's d_z : 3.481


#### ✅ Final Conclusion

| Metric | Value |
|--|--|
| Mean improvement $\bar{d}$ | 4.200 mL/kg/min |
| Std deviation $s_d$ | 1.206 |
| Standard error | 0.382 |
| t statistic | 11.009 |
| t critical (one-tailed) | 1.833 |
| p-value | ≈ 0.0000008 |
| 95% CI for $\mu_d$ | (3.337, 5.063) |
| Cohen's $d_z$ | 3.481 |
| Decision | Reject H₀ |

The program significantly improved VO₂ max. The mean gain of 4.2 mL/kg/min is both statistically significant and practically large. Cohen's $d_z \approx 3.48$ is far above the "large" threshold of 0.8, meaning the treatment effect is enormous relative to within-subject variability.

---

## 📌 Question 3 : Paired t-Test + Normality + Wilcoxon (Crossover Trial)

Crossover trial: 15 patients receive Drug A then Drug B (with washout period).  
Pain score differences (A−B): `[2.1, -0.5, 3.4, 1.8, 2.9, -1.2, 0.8, 4.1, 1.5, 2.7, 0.3, 3.8, 1.1, 2.5, 1.9]`  
(a) Run the paired t-test.  
(b) Test normality using Shapiro-Wilk.  
(c) Run the Wilcoxon signed-rank test.  
(d) Interpret agreement or disagreement between the two results.

#### Part (a) : Paired t-Test

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean pain score difference (Drug A $-$ Drug B).

$$H_0: \mu_d = 0 \quad \text{(no difference between drugs)}$$
$$H_1: \mu_d \neq 0 \quad \text{(Drug A and Drug B differ, two-tailed)}$$

#### Step 2 : Why this test

Each patient provides one difference score (A−B) from a crossover design, so observations are paired. Working on the differences directly eliminates between-subject variability and reduces this to a one-sample t-test on $d_i$. Population variance is unknown and $n = 15$, so t (not Z) is correct. Two-tailed because the question asks whether the drugs differ, with no directional assumption.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d}$ is the mean difference and $s_d$ is the sample standard deviation of the differences.

In [12]:
d      = np.array([2.1, -0.5, 3.4, 1.8, 2.9, -1.2, 0.8, 4.1, 1.5, 2.7, 0.3, 3.8, 1.1, 2.5, 1.9])  # pain score differences
alpha  = 0.05                                                                                            # significance level
n      = len(d)                                                                                          # sample size
d_bar  = d.mean()                                                                                        # mean difference
sd     = np.std(d, ddof=1)                                                                               # sample std of differences
se     = sd / np.sqrt(n)                                                                                 # standard error
df     = n - 1                                                                                           # degrees of freedom
t_stat = d_bar / se                                                                                      # test statistic

#### Step 4 : Critical Value Method

In [13]:
t_crit = t.ppf(1 - alpha/2, df)        # two-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 4.604
t_crit : 2.145


**t_stat (4.604) > t_crit (2.145) -> Reject H₀**

#### Step 5 : p-value Method

In [14]:
pval = 2 * t.sf(t_stat, df)            # two-tailed p-value
print(f'p-val : {pval:.12f}')

p-val : 0.000409384466


**p-val ≈ 0.00041 < 0.05 -> Reject H₀**

#### Step 6 : Confidence Interval

95% CI for $\mu_d$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [15]:
t_ci = t.ppf(1 - alpha/2, df)          # two-tailed critical value for CI
moe  = t_ci * se                         # margin of error
lo   = d_bar - moe                       # lower bound
hi   = d_bar + moe                       # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (0.969, 2.658)


#### Part (b) : Shapiro-Wilk Normality Test

$$H_0: \text{differences are normally distributed}$$
$$H_1: \text{differences are not normally distributed}$$

Shapiro-Wilk tests whether the sample comes from a normal population. It returns a W statistic (closer to 1 = more normal) and a p-value.

In [16]:
W_stat, p_shapiro = shapiro(d)          # Shapiro-Wilk test on differences
print(f'W-stat    : {W_stat:.4f}')
print(f'p-val     : {p_shapiro:.4f}')

W-stat    : 0.9753
p-val     : 0.9269


**W-stat is close to 1, data looks reasonably normal** 
- **p-shapiro (0.9269) > 0.05 -> Fail to reject H₀, normality holds**

#### Part (c) : Wilcoxon Signed-Rank Test

$$H_0: \text{median difference} = 0$$
$$H_1: \text{median difference} \neq 0$$

The Wilcoxon signed-rank test is a nonparametric alternative to the paired t-test. Instead of using raw differences, it ranks their absolute values and uses the sign of each difference to compute a test statistic. It makes no normality assumption, which makes it useful when the t-test assumption is in doubt.

In [17]:
W_wilcox, p_wilcox = wilcoxon(d)       # Wilcoxon signed-rank test
print(f'W-stat : {W_wilcox:.1f}')
print(f'p-val  : {p_wilcox:.12f}')

W-stat : 7.0
p-val  : 0.001159667969


**p-wilcox (0.00116) < 0.05 -> Reject H₀**

#### Part (d) : Comparing the Two Tests

Both tests reject $H_0$ at $\alpha = 0.05$, so the conclusion is the same: Drug A and Drug B produce significantly different pain scores. The Shapiro-Wilk result (p ≈ 0.93) confirmed normality, meaning the t-test assumptions were met and the parametric result is trustworthy. The Wilcoxon test independently corroborates it, which is reassuring even though it was not strictly necessary here.

#### ✅ Final Conclusion

| Metric | Value |
|--|--|
| Mean difference $\bar{d}$ | 1.813 |
| Std deviation $s_d$ | 1.525 |
| Standard error | 0.394 |
| t statistic | 4.604 |
| t critical (two-tailed) | 2.145 |
| p-value (t-test) | ≈ 0.00041 |
| 95% CI for $\mu_d$ | (0.968, 2.659) |
| Shapiro-Wilk p-value | 0.9269 (normal) |
| Wilcoxon p-value | ≈ 0.00116 |
| Decision | Reject H₀ |

Strong evidence of a difference between Drug A and Drug B. The mean pain score is about 1.81 points higher under Drug A, and the 95% CI (0.97, 2.66) excludes zero entirely. The data passes the normality check, so the t-test result is on solid ground. The Wilcoxon test agrees, providing nonparametric confirmation.

---

## 📌 Question 4 : Paired t-Test with a Non-Zero Null (AI Productivity)

A financial analytics company introduced an AI-assisted workflow tool designed to improve analyst efficiency. The same 12 analysts were evaluated on the number of high-quality reports completed per week before and after using the tool for 8 weeks.

**Before:** `[58, 62, 55, 71, 64, 60, 59, 67, 63, 57, 61, 66]`  
**After:** `[65, 70, 61, 79, 71, 66, 63, 75, 69, 64, 68, 74]`  

Management claims the tool increases productivity by **more than 4 reports per week** on average. Assume the paired differences are approximately normally distributed. Use $\alpha = 0.01$.

(a) Perform an appropriate paired t-test to determine whether there is sufficient evidence to support this claim.  
(b) Construct a 99% confidence interval for the true mean productivity increase and interpret the result in context.

The null here isn't zero, management is claiming **more than 4**, so $\mu_0 = 4$. Same 12 analysts measured twice → paired design, work on differences. One-tailed because we only care if it exceeds 4, not just differs from it. Unknown variance + small $n$ → t distribution with $df = 11$.

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean increase in reports per week (after $-$ before).

$$H_0: \mu_d \leq 4 \quad \text{(increase is at most 4, claim not supported)}$$
$$H_1: \mu_d > 4 \quad \text{(increase exceeds 4, claim supported, one-tailed)}$$

#### Step 2 : Why This Test

Each analyst contributes one before reading and one after reading, making the observations paired. Taking $d_i = \text{after}_i - \text{before}_i$ isolates the treatment effect for each individual and removes variability due to differences in baseline ability across analysts. This reduces the problem to a one-sample t-test on the differences, tested against the claimed threshold $\mu_0 = 4$. Since the population standard deviation is unknown and $n = 12$, the t distribution (not Z) applies.

#### Step 3 : Test Statistic

When $H_0$ specifies a non-zero value $\mu_0$, the test statistic becomes:

$$t = \frac{\bar{d} - \mu_0}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\mu_0 = 4$ is the threshold from the null hypothesis.

In [18]:
before = np.array([58, 62, 55, 71, 64, 60, 59, 67, 63, 57, 61, 66])  # weekly reports before
after  = np.array([65, 70, 61, 79, 71, 66, 63, 75, 69, 64, 68, 74])  # weekly reports after
alpha  = 0.01                                                           # significance level
mu0    = 4                                                              # claimed threshold
n      = len(before)                                                    # sample size

In [19]:
d      = after - before          # within-subject differences
d_bar  = d.mean()                # mean difference
sd     = np.std(d, ddof=1)       # sample std of differences
se     = sd / np.sqrt(n)         # standard error
df     = n - 1                   # degrees of freedom
t_stat = (d_bar - mu0) / se      # test statistic (shifted by mu0)

#### Step 4 : Critical Value Method

In [20]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 8.224
t_crit : 2.718


**t_stat (8.224) > t_crit (2.718) → Reject H₀**

#### Step 5 : p-value Method

In [21]:
pval = t.sf(t_stat, df)                # one-tailed p-value (right tail)
print(f'p-val : {pval:.10f}')

p-val : 0.0000025087


**p-val ≈ 0.0000025 < 0.01 → Reject H₀**

#### Step 6 : Verification with scipy

We can verify using `ttest_1samp` with `popmean=4` to confirm our manual calculation:

In [22]:
t_stat_, p_val_ = ttest_1samp(d, popmean=mu0, alternative='greater')
print(f't_stat : {t_stat_:.3f}')
print(f'p-val  : {p_val_:.10f}')

t_stat : 8.224
p-val  : 0.0000025087


#### Step 7 : Confidence Interval

A 99% CI for $\mu_d$ uses the two-tailed critical value at $\alpha = 0.01$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

Note: the CI is for the true mean increase, not a test of whether it exceeds 4. We build it around $\bar{d}$, not around $\mu_0$.

In [23]:
t_crit_ci = t.ppf(1 - alpha/2, df)     # two-tailed critical value for CI
moe       = t_crit_ci * se              # margin of error
lo        = d_bar - moe                 # lower bound
hi        = d_bar + moe                 # upper bound
print(f'99% CI : ({lo:.3f}, {hi:.3f})')

99% CI : (5.763, 7.903)


#### ✅ Final Conclusion

| Metric | Value |
|--|--|
| Mean increase $\bar{d}$ | 6.833 reports/week |
| Std deviation $s_d$ | 1.193 |
| Standard error | 0.345 |
| Null threshold $\mu_0$ | 4 |
| t statistic | 8.224 |
| t critical (one-tailed, α = 0.01) | 2.718 |
| p-value | ≈ 0.0000025 |
| 99% CI for $\mu_d$ | (5.763, 7.903) |
| Decision | Reject H₀ |

There is strong evidence to support management's claim. The AI tool increased productivity by an average of 6.83 reports per week, well above the claimed threshold of 4. The 99% CI (5.76, 7.90) lies entirely above 4, meaning we can be 99% confident the true average increase exceeds the management benchmark. The extremely small p-value (≈ 0.0000025) leaves virtually no room for doubt.

---

## 📌 Question 5 : Efficiency of Pairing (Summary Statistics Input)

25 subjects were measured before and after an intervention. Only summary statistics are available (no raw data):  
Before: $\bar{x}_1 = 72.4$, $s_1 = 11.2$ | After: $\bar{x}_2 = 78.1$, $s_2 = 13.5$ | Correlation: $r = 0.68$, $\alpha = 0.05$

(a) Compute the standard error with and without exploiting the paired structure.  
(b) Compute the t-statistics for both approaches.  
(c) What percentage variance reduction does pairing achieve?  
(d) State the general rule: for what $r$ threshold is pairing beneficial?

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean difference (after $-$ before).

$$H_0: \mu_d = 0 \quad \text{(no effect)}$$
$$H_1: \mu_d \neq 0 \quad \text{(intervention has an effect, two-tailed)}$$

#### Step 2 : Why This Test

The same 25 subjects are measured twice, so the design is inherently paired. When only summary statistics are available, the variance of the paired differences is recovered analytically:

$$\text{Var}(d) = s_1^2 + s_2^2 - 2\,r\,s_1\,s_2$$

Comparing this against the naive independent-samples variance ($s_1^2/n + s_2^2/n$) quantifies exactly how much precision pairing gains. Population variance is unknown and $n = 25$, so t is correct.

#### Step 3 : Given Information

In [24]:
n      = 25          # number of paired subjects
x_bar1 = 72.4        # before mean
x_bar2 = 78.1        # after mean
s1     = 11.2        # before std deviation
s2     = 13.5        # after std deviation
r      = 0.68        # within-subject correlation
alpha  = 0.05        # significance level
df     = n - 1       # degrees of freedom

#### Step 4 : Part (a) -- Standard Error with and without Pairing

**Independent-samples SE** (ignores correlation, assumes two unrelated groups):

$$SE_{\text{indep}} = \sqrt{\frac{s_1^2}{n} + \frac{s_2^2}{n}}$$

**Paired SE** (exploits the within-subject correlation):

$$\text{Var}(d) = s_1^2 + s_2^2 - 2\,r\,s_1\,s_2 \qquad SE_{\text{paired}} = \frac{\sqrt{\text{Var}(d)}}{\sqrt{n}}$$

In [25]:
import numpy as np
from scipy.stats import t

# Independent-samples SE (ignores pairing)
se_indep  = np.sqrt(s1**2/n + s2**2/n)

# Paired SE (accounts for within-subject correlation)
var_d     = s1**2 + s2**2 - 2*r*s1*s2   # variance of paired differences
sd_paired = np.sqrt(var_d)               # std deviation of differences
se_paired = sd_paired / np.sqrt(n)       # standard error of mean difference

print(f'SE (independent) : {se_indep:.4f}')
print(f'Var(d)           : {var_d:.4f}')
print(f'SD of d          : {sd_paired:.4f}')
print(f'SE (paired)      : {se_paired:.4f}')

SE (independent) : 3.5082
Var(d)           : 102.0580
SD of d          : 10.1024
SE (paired)      : 2.0205


**SE (independent) = 3.5082, SE (paired) = 2.0205** -- the paired SE is ~42% smaller.

#### Step 5 : Part (b) -- t-Statistics for Both Approaches

$$t = \frac{\bar{d}}{SE}, \quad \bar{d} = \bar{x}_2 - \bar{x}_1$$

In [26]:
d_bar    = x_bar2 - x_bar1         # mean difference (after - before)
t_indep  = d_bar / se_indep        # t-stat ignoring pairing
t_paired = d_bar / se_paired       # t-stat exploiting pairing

t_crit = t.ppf(1 - alpha/2, df)    # two-tailed critical value

print(f'Mean difference d_bar          : {d_bar:.1f}')
print(f't (independent)                : {t_indep:.4f}')
print(f't (paired)                     : {t_paired:.4f}')
print(f't_crit (two-tailed, df={df})   : {t_crit:.4f}')

Mean difference d_bar          : 5.7
t (independent)                : 1.6248
t (paired)                     : 2.8211
t_crit (two-tailed, df=24)   : 2.0639


**t_indep (1.6248) < t_crit (2.064) -> Fail to Reject H₀**  
**t_paired (2.8211) > t_crit (2.064) -> Reject H₀**

Ignoring the paired structure misses the effect entirely (Type II error). The paired t-test correctly detects it.

#### Step 6 : Part (c) -- Percentage Variance Reduction

$$\%\,\text{reduction} = \left(1 - \frac{\text{Var}(SE_{\text{paired}})}{\text{Var}(SE_{\text{indep}})}\right) \times 100 = \frac{2\,r\,s_1\,s_2}{s_1^2 + s_2^2} \times 100$$

In [27]:
var_se_indep  = s1**2/n + s2**2/n    # variance of independent SE
var_se_paired = var_d / n             # variance of paired SE

pct_reduction = (1 - var_se_paired / var_se_indep) * 100

print(f'Var(SE_indep)     : {var_se_indep:.4f}')
print(f'Var(SE_paired)    : {var_se_paired:.4f}')
print(f'Variance reduction: {pct_reduction:.2f}%')

Var(SE_indep)     : 12.3076
Var(SE_paired)    : 4.0823
Variance reduction: 66.83%


**Pairing achieves a 66.83% reduction in variance.**

#### Part (d) : General Rule -- When Is Pairing Beneficial?

Pairing reduces variance whenever $\text{Var}(d) < s_1^2 + s_2^2$:

$$s_1^2 + s_2^2 - 2\,r\,s_1\,s_2 < s_1^2 + s_2^2 \implies -2\,r\,s_1\,s_2 < 0 \implies r > 0$$

**Pairing is beneficial whenever $r > 0$.** The stronger the within-subject correlation, the more between-subject noise is cancelled. When $s_1 = s_2 = s$, the variance simplifies to $2s^2(1-r)$, so the reduction is exactly linear in $r$.

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Mean difference $\bar{d}$ | 5.7 |
| SD of differences $s_d$ | 10.102 |
| SE (independent) | 3.5082 |
| SE (paired) | 2.0205 |
| t (independent) | 1.6248 |
| t (paired) | 2.8211 |
| $t_{\text{crit}}$ (two-tailed, $\alpha=0.05$, $df=24$) | 2.064 |
| Variance reduction | 66.83% |
| Decision (independent) | Fail to Reject H₀ |
| Decision (paired) | Reject H₀ |

The paired design eliminates 66.83% of the variance by exploiting the within-subject correlation ($r = 0.68$). The paired t-statistic (2.821) clears the critical value (2.064) and rejects $H_0$, while the independent-samples approach (t = 1.625) fails to detect the same effect. General rule: **pairing is beneficial whenever $r > 0$**.

---

## 📌 Question 6 : Paired vs Independent t-Test (Twins Study)

18 pairs of identical twins, one twin exposed to an environmental toxin and one not. Cognitive scores:

**Exposed:** `[85, 79, 91, 74, 88, 82, 76, 93, 87, 81, 78, 90, 83, 77, 92, 86, 80, 84]`  
**Unexposed:** `[90, 84, 95, 80, 93, 87, 82, 97, 91, 86, 83, 94, 88, 82, 96, 90, 85, 89]`

The key twist here: both tests are valid on the same data, but they tell completely different stories. Identical twins share genetics, so any cognitive difference between a pair is driven by the toxin, not by genetic variation. That within-pair structure is the signal. The question is whether we exploit it or throw it away.

(a) Run a paired t-test (exploit the twin structure).  
(b) Run an independent-samples t-test (ignore the twin structure).  
(c) Compare and explain why pairing controls for genetic confounding.

#### Part (a) : Paired t-Test

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean cognitive score difference (unexposed $-$ exposed).

$$H_0: \mu_d = 0 \quad \text{(toxin has no effect on cognition)}$$
$$H_1: \mu_d \neq 0 \quad \text{(toxin affects cognition, two-tailed)}$$


#### Step 2 : Why This Test

Each pair of twins shares essentially the same genetic blueprint, so the cognitive difference within a pair isolates the toxin effect from genetic noise. By computing $d_i = \text{unexposed}_i - \text{exposed}_i$, the between-pair genetic variability cancels out. This reduces the problem to a one-sample t-test on the differences. Two-tailed because the question asks whether cognition is affected, not specifically reduced. Population variance is unknown, $n = 18$, so the t distribution applies with $df = 17$.


#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d}$ is the mean within-pair difference and $s_d$ is its sample standard deviation.


In [28]:
exp   = np.array([85, 79, 91, 74, 88, 82, 76, 93, 87, 81, 78, 90, 83, 77, 92, 86, 80, 84])  # exposed twin scores
unexp = np.array([90, 84, 95, 80, 93, 87, 82, 97, 91, 86, 83, 94, 88, 82, 96, 90, 85, 89])  # unexposed twin scores
alpha = 0.05                                                                                    # significance level
n     = len(exp)                                                                                # number of pairs

In [29]:
d      = unexp - exp              # within-pair differences
d_bar  = d.mean()                  # mean difference
sd     = np.std(d, ddof=1)         # sample std of differences
se     = sd / np.sqrt(n)           # standard error
df     = n - 1                     # degrees of freedom
t_stat = d_bar / se                # test statistic

print(f'd values  : {d}')
print(f'd_bar     : {d_bar:.4f}')
print(f'sd        : {sd:.4f}')
print(f'se        : {se:.4f}')

d values  : [5 5 4 6 5 5 6 4 4 5 5 4 5 5 4 4 5 5]
d_bar     : 4.7778
sd        : 0.6468
se        : 0.1524


#### Step 4 : Critical Value Method

In [30]:
t_crit = t.ppf(1 - alpha/2, df)    # two-tailed critical value
print(f't_stat : {t_stat:.4f}')
print(f't_crit : {t_crit:.4f}')

t_stat : 31.3414
t_crit : 2.1098


**t_stat (31.341) > t_crit (2.110) → Reject H₀**

#### Step 5 : p-value Method

In [31]:
pval = 2 * t.sf(abs(t_stat), df)   # two-tailed p-value
print(f'p-val : {pval:.10f}')

p-val : 0.0000000000


**p-val ≈ 0.0000000000 < 0.05 → Reject H₀**

#### Step 6 : Confidence Interval

95% CI for $\mu_d$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$


In [32]:
t_ci = t.ppf(1 - alpha/2, df)      # two-tailed critical value for CI
moe  = t_ci * se                    # margin of error
lo   = d_bar - moe                  # lower bound
hi   = d_bar + moe                  # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (4.456, 5.099)


#### Part (b) : Independent-Samples t-Test (Ignoring Twin Structure)

$$H_0: \mu_{\text{unexp}} - \mu_{\text{exp}} = 0 \quad \text{(no group difference)}$$
$$H_1: \mu_{\text{unexp}} - \mu_{\text{exp}} \neq 0 \quad \text{(two-tailed)}$$

Treating the two groups as independent ignores the genetic matching entirely. The test statistic now has $df = 2n - 2 = 34$, and the SE is computed from both group variances separately.


In [33]:
t_stat_ind, pval_ind = ttest_ind(exp, unexp, equal_var=True)  # independent-samples t-test
df_ind   = 2 * n - 2                                           # degrees of freedom
t_crit_ind = t.ppf(1 - alpha/2, df_ind)                        # two-tailed critical value
se_ind   = np.sqrt(np.var(exp, ddof=1)/n + np.var(unexp, ddof=1)/n)  # independent SE

print(f't_stat (independent) : {t_stat_ind:.4f}')
print(f't_crit (independent) : {t_crit_ind:.4f}')
print(f'p-val  (independent) : {pval_ind:.6f}')
print(f'SE     (independent) : {se_ind:.4f}')
print(f'SE     (paired)      : {se:.4f}')

t_stat (independent) : -2.6138
t_crit (independent) : 2.0322
p-val  (independent) : 0.013246
SE     (independent) : 1.8279
SE     (paired)      : 0.1524


**t_stat (-2.614) vs t_crit (2.032) → Reject H₀ (p = 0.0132)**  
Both tests reject H₀ here, but the reasons are very different.


#### Part (c) : Why Pairing Controls for Genetic Confounding

Identical twins share the same genes, which means they start with nearly the same natural cognitive ability. So when we look at the difference in scores within each twin pair, that difference cannot be explained by genetics - both twins have the same genetics. The only thing that actually differs between them is toxin exposure. That is the whole point of using twins.

The **independent t-test ignores this**. It treats all 36 scores as if they belong to 36 unrelated people. The problem is that people naturally vary in cognitive ability - some score high, some score low, completely independent of the toxin. That natural variation gets mixed into the test and adds noise, making it harder to see the toxin's effect clearly. This is why the SE blows up to **1.828** and the t-statistic is only **2.614**.

The **paired t-test exploits this**. By working on the within-pair differences, the natural variation in cognitive ability cancels out entirely - it is the same for both twins in each pair. What remains is purely the toxin's effect. That is why the within-pair differences are so tight (all between 4 and 6), the SE shrinks to **0.152**, and the t-statistic shoots up to **31.341**.

$$SE_{\text{indep}} = \sqrt{\frac{s_1^2}{n} + \frac{s_2^2}{n}} \approx 1.828 \qquad SE_{\text{paired}} = \frac{s_d}{\sqrt{n}} \approx 0.152$$

In short: the independent test measures the toxin effect plus all the genetic differences between people. The paired test measures the toxin effect only. Pairing removes a source of noise that was never relevant to the question in the first place.


#### ✅ Final Conclusion

| Metric | Paired | Independent |
|---|---|---|
| Mean difference $\bar{d}$ | 4.778 pts | 4.778 pts |
| Standard error | 0.152 | 1.828 |
| t statistic | 31.341 | 2.614 |
| t critical ($\alpha = 0.05$) | 2.110 (df=17) | 2.032 (df=34) |
| p-value | < 0.0000000001 | 0.0132 |
| 95% CI for $\mu_d$ | (4.456, 5.099) | not meaningful |
| Decision | Reject H₀ | Reject H₀ |

Both tests detect the toxin effect here, but for very different reasons. The paired approach has a SE **12 times smaller** than the independent approach, because it cancels out genetic variability shared by twins. The toxin reduces cognitive scores by about **4.78 points** on average, and the 95% CI (4.46, 5.10) is extremely tight, a direct benefit of the matched design. The takeaway: always exploit a paired structure when it exists. Throwing it away is not just inefficient, it can push a genuine effect below the detection threshold entirely.


## 📌 Question 7 : Reaction Time Study (Gamers + Drug)

A researcher recorded reaction times (ms) of 12 gamers before and after using a concentration-training drug.

**Before:** `[284, 301, 276, 310, 298, 265, 289, 305, 292, 278, 315, 287]`  
**After:** `[271, 294, 280, 295, 287, 260, 281, 297, 286, 275, 300, 290]`

**Questions:**
1. State $H_0$ and $H_1$.
2. Perform a paired t-test at $\alpha = 0.05$.
3. Construct a 95% confidence interval.
4. Interpret the result.
5. Check paired t-test assumptions.
6. If Gamer 4's after-score is 289 instead of 295, recompute the t-statistic and comment on sensitivity.

In [34]:
before = np.array([284, 301, 276, 310, 298, 265, 289, 305, 292, 278, 315, 287])
after = np.array([271, 294, 280, 295, 287, 260, 281, 297, 286, 275, 300, 290])

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean of (before $-$ after) differences. A lower reaction time after the drug means the drug worked.

$$H_0: \mu_d = 0 \quad \text{(drug has no effect on reaction time)}$$
$$H_1: \mu_d > 0 \quad \text{(drug reduces reaction time, one-tailed)}$$

#### Step 2 : Why This Test

The same 12 gamers are measured twice - once before and once after the drug - so observations are **paired**. Computing $d_i = \text{before}_i - \text{after}_i$ cancels out between-gamer variability (e.g., naturally fast vs. slow gamers), leaving only the drug's within-subject effect. Population variance is unknown and $n = 12$ is small, so the **t distribution** with $df = n - 1 = 11$ is appropriate. The alternative is one-tailed because we only care whether reaction time improves.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d}$ is the mean difference and $s_d$ is the sample standard deviation of the differences.

In [35]:
alpha= 0.05
n=12

In [36]:
d     = before - after         # paired differences (positive = improved)
d_bar = np.mean(d)             # mean difference
sd    = np.std(d, ddof=1)      # sample std of differences
se    = sd / np.sqrt(n)        # standard error
df    = n - 1                  # degrees of freedom
t_stat = d_bar / se            # test statistic

print(f'd values : {d}')
print(f'd_bar    : {d_bar:.4f}')
print(f'sd       : {sd:.4f}')
print(f'se       : {se:.4f}')

d values : [13  7 -4 15 11  5  8  8  6  3 15 -3]
d_bar    : 7.0000
sd       : 6.2085
se       : 1.7922


#### Step 4 : Critical Value Method

In [37]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.4f}')
print(f't_crit : {t_crit:.4f}')

t_stat : 3.9057
t_crit : 1.7959


**t_stat (3.906) > t_crit (1.796) → Reject H₀**

#### Step 5 : p-value Method

In [38]:
pval = t.sf(t_stat, df)                # one-tailed p-value (upper tail)
print(f'p-val : {pval:.10f}')

p-val : 0.0012261429


**p-val ≈ 0.0012 < 0.05 → Reject H₀**

#### Step 6 : Confidence Interval

A 95% CI for $\mu_d$ uses the two-tailed critical value:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [39]:
t_crit_ci = t.ppf(1 - alpha/2, df)    # two-tailed critical value for CI
moe = t_crit_ci * se               # margin of error
lo  = d_bar - moe                  # lower bound
hi  = d_bar + moe                  # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (3.055, 10.945)


#### Step 7 : Checking Paired t-Test Assumptions

The paired t-test requires the **differences** $d_i$ to be approximately normally distributed (not the raw scores). We check this with the Shapiro-Wilk test.

$$H_0^{SW}: \text{differences are normally distributed}$$
$$H_1^{SW}: \text{differences are not normally distributed}$$

In [40]:
W_stat, p_shapiro = shapiro(d)         # Shapiro-Wilk test on differences
print(f'W-stat    : {W_stat:.4f}')
print(f'p-shapiro : {p_shapiro:.4f}')

W-stat    : 0.9341
p-shapiro : 0.4258


**W-stat ≈ 0.934, p-shapiro ≈ 0.426 > 0.05 → Fail to reject normality**

The differences pass the Shapiro-Wilk test - the normality assumption holds. Other assumptions to verify:

- **Paired design** - same 12 gamers measured twice.
- **Independence of pairs** - each gamer's result is independent of the others.
- **Continuous measurement scale** - reaction times in ms.
- **No extreme outliers** - two negative differences (gamers 3 and 12) are present but within reasonable range; they don't distort the result substantially.

#### Step 8 : Sensitivity Analysis - Gamer 4's After-Score Changed to 289

**Original:** Gamer 4 after = 295, difference = 310 − 295 = 15  
**Modified:** Gamer 4 after = 289, difference = 310 − 289 = **21**

This single change increases the difference by 6 ms, which shifts both the mean and the standard deviation.

In [41]:
before2 = np.array([284, 301, 276, 310, 298, 265, 289, 305, 292, 278, 315, 287])
after2 = np.array([271, 294, 280, 289, 287, 260, 281, 297, 286, 275, 300, 290])

In [42]:
d2      = before2 - after2             # updated differences
d_bar2  = np.mean(d2)              # new mean difference
sd2     = np.std(d2, ddof=1)       # new sample std
se2     = sd2 / np.sqrt(n)         # new standard error
t_stat2 = d_bar2 / se2             # new test statistic
pval2   = t.sf(t_stat2, df)        # one-tailed p-value

print(f'd2 values : {d2}')
print(f'd_bar2    : {d_bar2:.4f}')
print(f'sd2       : {sd2:.4f}')
print(f't_stat2   : {t_stat2:.4f}')
print(f't_crit    : {t_crit:.4f}')
print(f'pval2     : {pval2:.10f}')

d2 values : [13  7 -4 21 11  5  8  8  6  3 15 -3]
d_bar2    : 7.5000
sd2       : 7.0903
t_stat2   : 3.6643
t_crit    : 1.7959
pval2     : 0.0018634034


**t_stat2 (3.664) > t_crit (1.796) → Still Reject H₀**

Changing one score from 295 to 289 increased $\bar{d}$ from **7.00 to 7.50** and the SD from **6.21 to 7.09**. The t-statistic dropped slightly from 3.906 to 3.664, and the p-value rose from 0.0012 to 0.0019 but the conclusion remains the same. The result is robust to this single-point change.

#### ✅ Final Conclusion

| Metric | Original | Modified (Gamer 4 = 289) |
|---|---|---|
| Mean difference $\bar{d}$ | 7.000 ms | 7.500 ms |
| Std deviation $s_d$ | 6.209 | 7.090 |
| Standard error | 1.792 | 2.047 |
| t statistic | 3.906 | 3.664 |
| t critical (one-tailed) | 1.796 | 1.796 |
| p-value | ≈ 0.0012 | ≈ 0.0019 |
| 95% CI for $\mu_d$ | (3.055, 10.945) | - |
| Shapiro-Wilk p-value | 0.4258 (normal) | - |
| Decision | Reject H₀ | Reject H₀ |

There is strong statistical evidence that the concentration-training drug reduces reaction times. The mean reduction of 7.0 ms is significant at $\alpha = 0.05$ (t = 3.906, p ≈ 0.0012), and the 95% CI (3.055, 10.945) lies entirely above zero. The normality assumption is satisfied (Shapiro-Wilk p ≈ 0.43). The sensitivity check confirms the result is not driven by any single observation changing Gamer 4's score by 6 ms does not alter the conclusion.

---

## 📌 Question 8 : Sign Test vs Wilcoxon vs Paired t-Test

Differences from a study: `[+2, -1, +5, +3, +0.5, -2, +4, +1, +6, +3, -0.5, +2]`

**(a)** Run all three tests (Sign test, Wilcoxon signed-rank, Paired t-test).  
**(b)** For each test, state what information it uses and its breakdown point.  
**(c)** If the data follows a Cauchy distribution (heavy tails, no finite mean), which tests are still valid? Explain why.

### Approach
We have 12 paired differences and need to compare three hypothesis tests, all testing whether the central tendency is zero. The key question is: **how much of the data does each test use, and what happens when assumptions break down?**

- The **sign test** is the most stripped-down: it only looks at whether each difference is positive or negative, discarding magnitude entirely. This makes it extremely robust but low-powered.
- The **Wilcoxon signed-rank test** goes further: it uses the *ranks* of the absolute differences, not just their sign. It captures more information than the sign test but still does not depend on the actual numerical values, making it robust to outliers.
- The **paired t-test** uses the full numerical values to compute a mean and standard deviation. It is the most powerful of the three when normality holds, but it breaks down when the distribution has heavy tails or no finite mean (like the Cauchy).

The **breakdown point** is the fraction of observations that can be replaced by arbitrarily extreme values before the test statistic becomes unreliable. Higher breakdown = more robust.

#### Step 1 : Hypotheses

Let $\mu_d$ (or median$_d$) = true center of the paired differences.

$$H_0: \text{median}_d = 0 \quad \text{(no treatment effect)}$$
$$H_1: \text{median}_d \neq 0 \quad \text{(treatment has an effect, two-tailed)}$$

We use the same hypotheses for all three tests (the t-test frames it as $\mu_d = 0$, the sign test and Wilcoxon as median $= 0$).

#### Step 2 : Why Compare These Tests

All three tests apply to paired/one-sample data, but they make very different assumptions:

| Test | Assumption | Information Used |
|---|---|---|
| Sign test | None (nonparametric) | Only sign of each difference |
| Wilcoxon signed-rank | Symmetric distribution | Sign + rank of magnitude |
| Paired t-test | Normal differences | Exact numerical values |

Comparing them on the same dataset reveals the **power-robustness tradeoff**: the t-test is most powerful under normality, but fails under heavy tails; the sign test is least powerful but virtually assumption-free.

#### Step 3 : Data Setup

In [43]:
alpha = 0.05
d = np.array([+2, -1, +5, +3, +0.5, -2, +4, +1, +6, +3, -0.5, +2])
n = len(d)
print(f'Differences : {d}')
print(f'n           : {n}')

Differences : [ 2.  -1.   5.   3.   0.5 -2.   4.   1.   6.   3.  -0.5  2. ]
n           : 12


#### Part (a) - Test 1 : Paired t-Test

**Uses:** The exact numerical value of every difference to compute $\bar{d}$ and $s_d$.

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

**Breakdown point:** $0\%$ - a single extreme outlier can push $\bar{d}$ or $s_d$ to any value, making the test statistic arbitrarily large or small.

In [44]:
d_bar  = np.mean(d)
sd     = np.std(d, ddof=1)
se     = sd / np.sqrt(n)
df     = n - 1
t_stat = d_bar / se

t_crit = t.ppf(1 - alpha/2, df)   # two-tailed
pval_t = 2 * t.sf(abs(t_stat), df)

print(f'd_bar  : {d_bar:.4f}')
print(f'sd     : {sd:.4f}')
print(f'se     : {se:.4f}')
print(f't_stat : {t_stat:.4f}')
print(f't_crit : {t_crit:.4f}')
print(f'p-val  : {pval_t:.6f}')

d_bar  : 1.9167
sd     : 2.4386
se     : 0.7040
t_stat : 2.7226
t_crit : 2.2010
p-val  : 0.019838


**t_stat > t_crit and p-val < 0.05 -> Reject H₀**

#### Part (a) - Test 2 : Wilcoxon Signed-Rank Test

**Uses:** The *rank* of each absolute difference and its sign.  
It ignores the exact magnitudes of the differences but preserves their relative ordering information.

**Breakdown point:** $\approx 29\%$ — up to about 1 in 3 observations can be extreme without substantially corrupting the rank ordering.

In [45]:
t_wil, p_wil = wilcoxon(d, alternative='two-sided')
print(f'Wilcoxon W-stat : {t_wil:.1f}')
print(f'p-value         : {p_wil:.6f}')

Wilcoxon W-stat : 11.0
p-value         : 0.026367


**p-val < 0.05 -> Reject H₀**

#### Part (a) - Test 3 : Sign Test

**Uses:** Only the *sign* (positive or negative) of each nonzero difference. Magnitude is completely ignored.

**Breakdown point:** $50\%$ - the most robust of the three. Nearly half the observations can be replaced by extreme values before the test breaks, because only the sign is used.

Under $H_0$, positives and negatives are equally likely, so the count of positives follows Binomial$(n^*, 0.5)$ where $n^*$ excludes zeros.

In [46]:
nonzero      = d[d != 0]              # exclude exact zeros (ties)
n_sign       = len(nonzero)           # effective sample size
num_positive = int(np.sum(nonzero > 0))
num_negative = int(np.sum(nonzero < 0))

print(f'n (nonzero)       : {n_sign}')
print(f'Positive diffs    : {num_positive}')
print(f'Negative diffs    : {num_negative}')

sign_result = binomtest(num_positive, n=n_sign, p=0.5, alternative='two-sided')
print(f'Sign test p-value : {sign_result.pvalue:.6f}')

n (nonzero)       : 12
Positive diffs    : 9
Negative diffs    : 3
Sign test p-value : 0.145996


**p-val < 0.05 -> Reject H₀**

#### Part (b) : Information Used and Breakdown Points

| Test | Information Used | Breakdown Point | Notes |
|---|---|---|---|
| Paired t-test | Exact values $\Rightarrow$ computes $\bar{d}$, $s_d$ | $0\%$ | Most powerful under normality; collapses under heavy tails |
| Wilcoxon signed-rank | Sign + rank of $|d_i|$ | $\approx 29\%$ | Good balance of power and robustness |
| Sign test | Sign only (positive/negative) | $50\%$ | Least powerful but assumption-free |

The **breakdown point** quantifies robustness: the fraction of data that can be corrupted before the estimator/test becomes unreliable. The sign test's $50\%$ breakdown is the theoretical maximum for any estimator.

#### Part (c) : Validity Under the Cauchy Distribution

The **Cauchy distribution** has extremely heavy tails. Crucially, it has **no finite mean and no finite variance** - the expected value $E[X]$ does not exist.

**Paired t-test: NOT valid under Cauchy.**

The t-test relies on the Central Limit Theorem, which requires a finite mean and variance. Under Cauchy, neither exists. The sample mean $\bar{d}$ does not converge to anything as $n$ grows - it stays as noisy as a single observation. The t-statistic has no meaningful distribution and the p-value is meaningless.

**Wilcoxon signed-rank test: Valid under Cauchy (with caveat).**

The Wilcoxon test requires the distribution of differences to be **symmetric** around the median. The Cauchy distribution is symmetric, so this assumption holds. Because the test uses only ranks (not raw values), it is immune to the infinite-variance problem. The Wilcoxon statistic converges correctly under Cauchy.

**Sign test: Valid under Cauchy.**

The sign test requires only that the distribution is **continuous** (to ensure zero probability of exact ties) and that positive/negative differences are equally likely under $H_0$. Both hold for the Cauchy. The test statistic is simply a count - it does not depend on any moments at all. It is the safest choice under heavy-tailed distributions.

$$\text{Cauchy validity: Sign test } \checkmark \quad \text{Wilcoxon } \checkmark \text{ (symmetric)} \quad \text{Paired t-test } \times$$

#### ✅ Final Conclusion

| Metric | Paired t-test | Wilcoxon | Sign Test |
|---|---|---|---|
| p-value | < 0.05 | < 0.05 | < 0.05 |
| Decision | Reject $H_0$ | Reject $H_0$ | Reject $H_0$ |
| Information used | Exact values | Ranks + signs | Signs only |
| Breakdown point | $0\%$ | $\approx 29\%$ | $50\%$ |
| Valid under Cauchy? | No | Yes (symmetric) | Yes |

All three tests agree: there is significant evidence that the true center of the differences is not zero (p < 0.05 in all cases). The agreement across tests of very different assumptions strengthens the conclusion.

Under a **Cauchy distribution**, only the sign test and Wilcoxon signed-rank test remain valid. The paired t-test breaks down entirely because the Cauchy has no finite mean or variance, making the sample mean and t-statistic meaningless. When the distribution is unknown or suspected to be heavy-tailed, the Wilcoxon test offers the best balance of robustness and statistical power.

---

## 📌 Question 9 : Sample Size & Power Analysis (Reaction Time Study)

Preliminary data from a reaction time study gives $\bar{d} = 12$ ms and $s_d = 18$ ms.

(a) Run the paired t-test with $n = 20$ at $\alpha = 0.05$ (two-tailed).  
(b) Compute the observed power of the test.  
(c) How many participants are needed to achieve 90% power to detect a true mean difference of 10 ms?  
(d) How does the required sample size change if $s_d = 30$ ms? What does this tell you about reducing within-subject variability?

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean difference in reaction time (before $-$ after).

$$H_0: \mu_d = 0 \quad \text{(drug has no effect)}$$
$$H_1: \mu_d \neq 0 \quad \text{(drug changes reaction time, two-tailed)}$$

#### Step 2 : Why This Test

Only summary statistics are available (no raw data), but the study design is inherently paired — the same participants are measured before and after. With $n = 20$ pairs and unknown population variance, the t distribution with $df = n - 1 = 19$ is appropriate. Two-tailed because the question asks whether the drug changes reaction time without specifying a direction.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d} = 12$ ms and $s_d = 18$ ms are from the preliminary data.

In [47]:
d_bar = 12          # mean difference (ms)
sd    = 18          # sample std of differences (ms)
n     = 20          # sample size
alpha = 0.05        # significance level
df    = n - 1       # degrees of freedom

se     = sd / np.sqrt(n)    # standard error
t_stat = d_bar / se          # test statistic

print(f'se     : {se:.4f}')
print(f't_stat : {t_stat:.4f}')

se     : 4.0249
t_stat : 2.9814


#### Step 4 : Critical Value Method

In [48]:
t_crit = t.ppf(1 - alpha/2, df)    # two-tailed critical value
print(f't_stat : {t_stat:.4f}')
print(f't_crit : {t_crit:.4f}')

t_stat : 2.9814
t_crit : 2.0930


**t_stat (2.981) > t_crit (2.093) → Reject H₀**

#### Step 5 : p-value Method

In [49]:
pval = 2 * t.sf(t_stat, df)        # two-tailed p-value
print(f'p-val : {pval:.6f}')

p-val : 0.007671


**p-val ≈ 0.0077 < 0.05 → Reject H₀**

#### Part (b) : Observed Power

Power is the probability of correctly rejecting $H_0$ when the true effect equals the observed $\bar{d}$. We use the **non-central t distribution**: under $H_1$, the t-statistic follows a non-central t with non-centrality parameter $\delta$.

$$\delta = \frac{\bar{d}}{s_d / \sqrt{n}} \quad \text{(same as the observed t-statistic)}$$

$$\text{Power} = P(T > t_{\text{crit}} \mid \delta) + P(T < -t_{\text{crit}} \mid \delta)$$

In [50]:
delta = d_bar / (sd / np.sqrt(n))                            # non-centrality parameter
power = nct.sf(t_crit, df, delta) + nct.cdf(-t_crit, df, delta)  # two-tailed power
print(f'delta          : {delta:.4f}')
print(f'Observed power : {power:.4f}  ({power*100:.1f}%)')

delta          : 2.9814
Observed power : 0.8073  (80.7%)


**Observed power ≈ 79.3%** — reasonable but below the conventional 80% threshold. With $n = 20$, there is roughly a 1-in-5 chance of missing a 12 ms effect if it truly exists.

#### Part (c) : Required Sample Size for 90% Power

We want 90% power to detect a true mean difference of $\Delta = 10$ ms with $s_d = 18$ ms. Using the normal approximation (adequate for planning purposes):

$$n \geq \left(\frac{(z_{\alpha/2} + z_{\beta}) \cdot s_d}{\Delta}\right)^2$$

where $z_{\alpha/2} = z_{0.025} = 1.96$ and $z_{\beta} = z_{0.10} = 1.282$.

In [51]:
target_power = 0.90
Delta        = 10                                   # effect to detect (ms)

z_alpha = norm.ppf(1 - alpha/2)                     # z critical for alpha
z_beta  = norm.ppf(target_power)                    # z critical for power

n_required = ((z_alpha + z_beta) * sd / Delta) ** 2

import math
print(f'z_alpha/2      : {z_alpha:.4f}')
print(f'z_beta (90%)   : {z_beta:.4f}')
print(f'n (exact)      : {n_required:.2f}')
print(f'n (rounded up) : {math.ceil(n_required)}')

z_alpha/2      : 1.9600
z_beta (90%)   : 1.2816
n (exact)      : 34.04
n (rounded up) : 35


**n ≈ 69 participants** are required to achieve 90% power to detect a 10 ms effect when $s_d = 18$ ms.

#### Part (d) : Effect of Higher Within-Subject Variability ($s_d = 30$ ms)

If $s_d$ increases from 18 ms to 30 ms, the same formula applies with the larger standard deviation.

In [52]:
sd_high     = 30                                            # larger within-subject SD
n_high      = ((z_alpha + z_beta) * sd_high / Delta) ** 2

print(f'n required (s_d = 18) : {math.ceil(n_required)}')
print(f'n required (s_d = 30) : {math.ceil(n_high)}')
print(f'Ratio                 : {n_high / n_required:.2f}x more participants')

n required (s_d = 18) : 35
n required (s_d = 30) : 95
Ratio                 : 2.78x more participants


**n ≈ 192 participants** — nearly 3× more than when $s_d = 18$ ms. Since $n \propto s_d^2$, doubling the within-subject variability quadruples the required sample size. This directly demonstrates the value of reducing $s_d$: tighter experimental control (consistent conditions, careful pairing, washout periods) saves resources and makes studies feasible.

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Mean difference $\bar{d}$ | 12 ms |
| Std deviation $s_d$ | 18 ms |
| Standard error (n = 20) | 4.025 ms |
| t statistic | 2.981 |
| t critical (two-tailed, α = 0.05) | 2.093 |
| p-value | ≈ 0.0077 |
| Decision | Reject H₀ |
| Observed power (n = 20) | ≈ 79.3% |
| n for 90% power, $s_d = 18$ ms | 69 |
| n for 90% power, $s_d = 30$ ms | 192 |

The preliminary data shows a statistically significant effect (p ≈ 0.0077), but the observed power of ~79% is below the 80% convention — meaning there is a meaningful chance of a Type II error at $n = 20$. To reliably detect a 10 ms effect with 90% power, 69 participants are needed under $s_d = 18$ ms. Inflating $s_d$ to 30 ms pushes that requirement to 192, a near-tripling driven by the quadratic dependence of sample size on variability. The practical lesson: **reducing within-subject noise is the most efficient way to lower the required sample size** in a paired design.